# StochX — Complete Python Tour\n\nThis notebook is a practical tour of the public StochX API in v0.2.0. It is organized around the mathematical objects exposed by the package and is meant to be executable from top to bottom.\n\nWe cover:\n1. package setup and validation\n2. discrete-time Markov chains (DTMC)\n3. homogeneous and non-homogeneous Poisson processes\n4. continuous-time Markov chains (CTMC) and uniformization\n5. birth-death processes\n6. finite probability spaces, random variables, partitions, and conditional expectation\n7. filtrations, martingales, stopping times, and stopped processes\n8. trajectory analysis and an end-to-end stochastic workflow

In [ ]:
# For a fresh environment, install the released package first:\n# %pip install -U stochx==0.2.0\n\nimport numpy as np\nimport stochx\n\nfrom stochx.stochastic import (\n    BirthDeathProcess,\n    ContinuousTimeMarkovChain,\n    FiniteProbabilitySpace,\n    Filtration,\n    MarkovChain,\n    Martingale,\n    NonHomogeneousPoissonProcess,\n    Partition,\n    PoissonProcess,\n    StoppingTime,\n)\nfrom stochx.stochastic.analysis import empirical_state_frequencies\n\nprint('StochX version:', stochx.__version__)

## 1. Validation first\n\nStochX validates stochastic matrices and generators when the objects are constructed. This is useful because numerical mistakes are caught at the model boundary instead of appearing later during simulation or analysis.

In [ ]:
# Invalid stochastic matrix: the second row does not sum to 1.\ntry:\n    MarkovChain([[0.7, 0.3], [0.4, 0.4]])\nexcept Exception as exc:\n    print(type(exc).__name__, ':', exc)\n\n# Invalid generator: the second row does not sum to 0.\ntry:\n    ContinuousTimeMarkovChain([[-1.0, 1.0], [1.0, 0.0]])\nexcept Exception as exc:\n    print(type(exc).__name__, ':', exc)

## 2. Discrete-time Markov chains\n\nA `MarkovChain` is created from a finite row-stochastic transition matrix `P`. State labels can be strings, while StochX keeps the numerical indexing internal. The public API covers transition powers, classification, stationarity, hitting/return quantities, and simulation.

In [ ]:
P = np.array([[0.7, 0.3],\n              [0.4, 0.6]], dtype=float)\n\ndtmc = MarkovChain(P, states=['A', 'B'])\n\nprint('states:', dtmc.states)\nprint('n_states:', dtmc.n_states)\nprint('P =\n', dtmc.transition_matrix)\nprint('P^5 =\n', dtmc.transition_matrix_at(5))\nprint('n_step_transition(5) =\n', dtmc.n_step_transition(5))\nprint('communicating classes:', dtmc.communicating_classes)\nprint('closed classes:', dtmc.closed_classes)\nprint('irreducible:', dtmc.is_irreducible)\nprint('aperiodic:', dtmc.is_aperiodic)\nprint('ergodic:', dtmc.is_ergodic)

In [ ]:
print('stationary_distribution:', dtmc.stationary_distribution())\nprint('all stationary distributions:', dtmc.stationary_distributions())\nprint('mean return time to A:', dtmc.mean_return_time('A'))\nprint('return probability of A:', dtmc.return_probability('A'))\nprint('first return P(T_A=3):', dtmc.first_return_probability('A', 3))\nprint('hitting probability A -> B:', dtmc.hitting_probability('A', 'B'))\nprint('expected hitting time A -> B:', dtmc.expected_hitting_time('A', 'B'))\nprint('accessible A -> B:', dtmc.accessible('A', 'B'))\nprint('communicate A <-> B:', dtmc.communicate('A', 'B'))

In [ ]:
mu0 = np.array([1.0, 0.0])\nprint('state distribution at n=4:', dtmc.state_distribution(mu0, 4))\nprint('Chapman-Kolmogorov P^2 P^3 =\n', dtmc.chapman_kolmogorov(2, 3))\nprint('absorbing A?:', dtmc.is_absorbing_state('A'))\n\npath = dtmc.simulate(10_000, initial_state='A', rng=np.random.default_rng(42))\nprint('first simulated states:', path[:10])\nprint('empirical frequencies:', empirical_state_frequencies(path, dtmc.states))

## 3. Homogeneous Poisson process\n\n`PoissonProcess` models a constant rate `lambda_ = rate` and exposes both analytic probabilities and simulation helpers.

In [ ]:
pp = PoissonProcess(rate=2.0)\nprint('rate:', pp.rate)\nprint('lambda_:', pp.lambda_)\nprint('P(N(3)=4):', pp.count_probability(4, 3.0))\nprint('P(N(3)-N(1)=3):', pp.increment_probability(3, 1.0, 3.0))\n\nrng = np.random.default_rng(42)\ninterarrivals = pp.interarrival_samples(5, rng=rng)\nprint('inter-arrivals:', interarrivals)\nprint('arrival times:', pp.arrival_times(5, rng=np.random.default_rng(42)))\nprint('simulated events up to t=5:', pp.simulate(5.0, rng=np.random.default_rng(42)))\nprint('conditional arrival times:', pp.conditional_arrival_times(4, 3.0, rng=np.random.default_rng(42)))

In [ ]:
pp2 = PoissonProcess(rate=3.0)\nsuperposed = pp.superpose(pp2)\nleft, right = pp.split(0.25)\nprint('superposed rate:', superposed.rate)\nprint('split rates:', left.rate, right.rate)

## 4. Non-homogeneous Poisson process\n\nHere the intensity is a callable `lambda(t)` and the mean function is `m(t) = integral_0^t lambda(s) ds`.

In [ ]:
intensity = lambda t: 1.0 + t\nmean = lambda t: t + 0.5 * t**2\nnhpp = NonHomogeneousPoissonProcess(intensity, mean_function=mean)\nprint('mean m(2):', nhpp.mean(2.0))\nprint('P(N(2)=3):', nhpp.count_probability(3, 2.0))\nprint('increment probability:', nhpp.increment_probability(2, 1.0, 2.0))\nprint('simulated events:', nhpp.simulate(3.0, rng=np.random.default_rng(42)))

## 5. Continuous-time Markov chains\n\nA `ContinuousTimeMarkovChain` is built from an infinitesimal generator `Q`. The package exposes the matrix-exponential route and Jensen uniformization as separate numerical paths.

In [ ]:
Q = np.array([[-2.0, 2.0],\n              [1.0, -1.0]], dtype=float)\n\nctmc = ContinuousTimeMarkovChain(Q, states=['A', 'B'])\nprint('states:', ctmc.states)\nprint('generator =\n', ctmc.generator)\nprint('generator_matrix alias =\n', ctmc.generator_matrix)\nprint('holding rates:', ctmc.holding_rates)\nprint('jump-chain matrix =\n', ctmc.jump_chain_matrix)\n\nP_exp = ctmc.transition_matrix(2.0)\nP_uni = ctmc.transition_matrix_uniformized(2.0)\nprint('P(2) by expm =\n', P_exp)\nprint('P(2) by uniformization =\n', P_uni)\nprint('max numerical difference:', np.max(np.abs(P_exp - P_uni)))

In [ ]:
print('state distribution:', ctmc.state_distribution([1.0, 0.0], 2.0))\nprint('Chapman-Kolmogorov check =\n', ctmc.chapman_kolmogorov(1.0, 2.0))\nprint('forward derivative =\n', ctmc.forward_derivative(1.0))\nprint('backward derivative =\n', ctmc.backward_derivative(1.0))\nprint('I + hQ =\n', ctmc.infinitesimal_transition_matrix(0.001))\nprint('holding rate of A:', ctmc.holding_rate('A'))\nprint('one holding time:', ctmc.holding_time('A', rng=np.random.default_rng(42)))\nprint('jump chain states:', ctmc.jump_chain().states)\nprint('communicating classes:', ctmc.communicating_classes())\nprint('stationary distribution:', ctmc.stationary_distribution())\nprint('stationary from jump chain:', ctmc.stationary_distribution_from_jump_chain())\nprint('mean return time:', ctmc.mean_return_time('A'))\nprint('long-run unit cost:', ctmc.long_run_cost([1.0, 3.0]))

In [ ]:
ctmc_path = ctmc.simulate(10.0, initial_state='A', rng=np.random.default_rng(42))\nprint('state at t=2:', ctmc_path.state_at(2.0))\nprint('occupation time of A:', ctmc_path.occupation_time('A', 10.0))\nprint('occupation fraction of A:', ctmc_path.occupation_fraction('A', 10.0))

## 6. Birth-death processes\n\n`BirthDeathProcess` makes birth and death rates explicit. Finite matrix construction uses `max_state`; the pure factories remain unbounded until you ask for a bounded generator representation.

In [ ]:
bd = BirthDeathProcess.linear(\n    birth_rate=0.2,\n    death_rate=0.1,\n    immigration=1.0,\n    max_state=6,\n)\n\nprint('lambda_3:', bd.birth_rate(3))\nprint('mu_3:', bd.death_rate(3))\nprint('generator Q =\n', bd.generator_matrix())\nprint('jump chain =\n', bd.jump_chain_matrix())\nprint('Kolmogorov derivative:', bd.kolmogorov_derivative([1, 0, 0, 0, 0, 0, 0]))\nprint('stationary weights:', bd.stationary_weights(7))\nprint('stationary distribution:', bd.stationary_distribution())\nprint('jump-chain states:', bd.jump_chain().states)\nprint('converted CTMC states:', bd.to_ctmc().states)

In [ ]:
pure_immigration = BirthDeathProcess.pure_immigration(2.0)\npure_birth = BirthDeathProcess.pure_birth(1.0)\npure_death = BirthDeathProcess.pure_death(1.0)\n\nprint('pure immigration rate at 3:', pure_immigration.birth_rate(3))\nprint('pure birth rate at 3:', pure_birth.birth_rate(3))\nprint('pure death rate at 3:', pure_death.death_rate(3))\nprint('pure birth P(X_3=2):', pure_birth.pure_birth_probability(2, 3.0))\nprint('pure death P(X_3=1 | X_0=3):', pure_death.pure_death_probability(1, 3.0, initial_population=3))

## 7. Finite probability space and random variables\n\nThe finite-probability API provides the common foundation for conditional expectation and the discrete-time martingale objects.

In [ ]:
space = FiniteProbabilitySpace(\n    outcomes=[0, 1, 2, 3],\n    probabilities=[0.25, 0.25, 0.25, 0.25],\n)\n\nX = space.random_variable([1.0, 2.0, 3.0, 4.0], name='X')\nY = space.random_variable([0.0, 0.0, 1.0, 1.0], name='Y')\n\nprint('outcomes:', space.outcomes)\nprint('probabilities:', space.probabilities)\nprint('P({1,2}):', space.probability_of({1, 2}))\nprint('X values:', X.array())\nprint('X support:', X.support)\nprint('E[X]:', X.expectation())\nprint('E[Y]:', Y.expected_value())\nprint('X+Y:', (X + Y).array())\nprint('X^2:', X.apply(lambda x: x**2, name='X2').array())

In [ ]:
G = Partition.generated_by(Y)\nG_from_blocks = Partition.from_blocks([{0, 1}, {2, 3}], space)\nprint('G blocks:', G.blocks)\nprint('G contains 2:', G.contains(2))\nprint('G refines itself:', G.refines(G_from_blocks))\n\nEX_given_Y = space.conditional_expectation_given(X, Y)\nEX_given_G = space.conditional_expectation(X, G)\nprint('E[X|Y]:', EX_given_Y.array())\nprint('E[X|G]:', EX_given_G.array())\nprint('E[X | event {1,2}]:', space.conditional_expectation_given_event(X, {1, 2}))\nprint('P({1,2} | {1,3}):', space.conditional_probability_given_event({1, 2}, {1, 3}))

In [ ]:
print('P(Y=1) and P(X=...):', space.probability_of({2, 3}), space.probability_of({0, 1}))\nprint('independent X,Y:', space.are_independent(X, Y))\nprint('independent partitions:', space.are_partitions_independent(G, G))\nprint('total expectation:', space.total_expectation(X, G))\nprint('total variance:', space.total_variance(X, G))\nprint('total covariance:', space.total_covariance(X, Y, G))\nprint('conditional variance:', space.conditional_variance(X, G).array())\nprint('conditional covariance:', space.conditional_covariance(X, Y, G).array())\nprint('L2 projection:', space.l2_projection(X, G).array())\nprint('characterization error:', space.conditional_characterization_error(X, G))

## 8. Filtrations and martingales\n\nWe now build a simple random walk on the same finite sample space and generate its natural filtration.

In [ ]:
eps1 = space.random_variable([1.0, 1.0, -1.0, -1.0], name='epsilon_1')\neps2 = space.random_variable([1.0, -1.0, 1.0, -1.0], name='epsilon_2')\nX0 = space.random_variable([0.0] * 4, name='X0')\nX1 = X0 + eps1\nX2 = X1 + eps2\n\nfiltration = Filtration.natural([X0, X1, X2])\nprint('number of filtration levels:', filtration.n_steps)\nprint('F_1:', filtration.at(1).blocks)\nprint('adapted:', filtration.is_adapted([X0, X1, X2]))\n\nmart = Martingale([X0, X1, X2], filtration)\nprint('M_0:', mart.value_at(0).array())\nprint('E[M_2 | F_0]:', mart.conditional_future(0, 2).array())\nprint('martingale residual at 1:', mart.martingale_residual(1))\nprint('is martingale:', mart.is_martingale())\nprint('is submartingale:', mart.is_submartingale())\nprint('is supermartingale:', mart.is_supermartingale())\nprint('expectations:', mart.expectations())

## 9. Stopping times and stopped processes\n\nThe most conservative finite example is a deterministic stopping time. A constant stopping time is always adapted to the filtration, which makes it a good first notebook example before experimenting with non-trivial stopping rules.

In [ ]:
T = StoppingTime.from_values(\n    space,\n    values=[2, 2, 2, 2],\n    filtration=filtration,\n)\nS = StoppingTime.from_values(\n    space,\n    values=[1, 1, 1, 1],\n    filtration=filtration,\n)\n\nprint('T values:', T.values)\nprint('min(T,S):', T.minimum(S).values)\nprint('max(T,S):', T.maximum(S).values)\nprint('T+S:', T.add(S).values)\n\nMT = mart.stopped(T)\nprint('stopped process sequence:', MT.sequence())\nprint('terminal stopped value:', MT.terminal_value())

## 10. A compact end-to-end workflow\n\nThe following pattern is a useful mental model for using StochX in real work:\n\n**define → validate → inspect → compute → simulate → analyze**.

In [ ]:
# 1. Define a DTMC\nmodel = MarkovChain([[0.8, 0.2], [0.3, 0.7]], states=['Healthy', 'Sick'])\n\n# 2. Inspect intrinsic structure\nprint('classes:', model.communicating_classes)\nprint('irreducible:', model.is_irreducible)\n\n# 3. Compute long-run behavior\npi = model.stationary_distribution()\nprint('stationary law:', pi)\n\n# 4. Simulate a long trajectory\ntrajectory = model.simulate(20_000, initial_state='Healthy', rng=np.random.default_rng(123))\n\n# 5. Compare simulation with theory\nempirical = empirical_state_frequencies(trajectory, model.states)\nprint('empirical:', empirical)\nprint('absolute error:', np.abs(empirical - pi))

## 11. Where to go next\n\nUse this notebook as the practical companion to the StochX API reference. When you need the mathematics, read the corresponding Course Material chapter; when you need exact constructor/property/method behavior, use the API pages.\n\nMain references inside this repository:\n- `docs/api/index.md` — API map\n- `docs/course_material.md` — course-material entry point\n- `examples/01_*.py` through `examples/07_*.py` — focused examples\n- this notebook — integrated interactive tour